In [5]:
import os
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm import tqdm

os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms.functional as TF

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
device: cuda:0
GPU: NVIDIA RTX A6000


In [6]:
DATA_ROOT = "/home/jpsong/imgProcessing_Solo/CelebAMask-HQ"  # prepared dataset root
OUT_DIR = "./runs_celebamaskhq_hyperparam_tuning_V2"    # output directory
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# 클래스 수 
NUM_CLASSES = 19

# 학습 파라미터
IMAGE_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 40
LR = 1e-4
WEIGHT_DECAY = 5e-4
NUM_WORKERS = 4
IGNORE_INDEX = 255

USE_AMP = True    
FREEZE_BACKBONE_EPOCHS = 2
INCLUDE_BG_IN_MIOU = False

In [7]:
LABELS = [
    "skin", "nose", "eye_g", "l_eye", "r_eye", "l_brow", "r_brow", "l_ear", "r_ear",
    "mouth", "u_lip", "l_lip", "hair", "hat", "ear_r", "neck_l", "neck", "cloth"
]
NAME2ID = {name: i + 1 for i, name in enumerate(LABELS)}  # 배경=0 (있어야됨)

def infer_anno_folder(img_id: int, chunk_size: int = 2000) -> str:
    return str(img_id // chunk_size)

def merge_masks_for_one(img_id: int, anno_root: Path) -> np.ndarray:
    subdir = anno_root / infer_anno_folder(img_id)
    out = None

    for name, lid in NAME2ID.items():
        mpath = subdir / f"{img_id:05d}_{name}.png"
        if not mpath.exists():
            continue
        m = np.array(Image.open(mpath))
        if m.ndim == 3: # RGB -> 첫 채널만 사용
            m = m[..., 0]
        m = (m > 0).astype(np.uint8)

        if out is None:
            out = np.zeros_like(m, dtype=np.uint8)

        out[m == 1] = lid

    if out is None:
        raise RuntimeError(f"No mask found for {img_id:05d} in {subdir}")
    return out

In [ ]:
# CELEB_ORIG_ROOT = "/home/jpsong/imgProcessing_Solo/CelebAMask-HQ" 

# ORIG_IMG_DIR = Path(CELEB_ORIG_ROOT) / "CelebA-HQ-img"
# ORIG_ANNO_DIR = Path(CELEB_ORIG_ROOT) / "CelebAMask-HQ-mask-anno"
# PREP_ROOT = Path(DATA_ROOT)
# (PREP_ROOT / "images").mkdir(parents=True, exist_ok=True)
# (PREP_ROOT / "masks").mkdir(parents=True, exist_ok=True)
# (PREP_ROOT / "splits").mkdir(parents=True, exist_ok=True)

# # 이미지 목록
# imgs = sorted([p for p in ORIG_IMG_DIR.glob("*") if p.suffix.lower() in [".jpg", ".png", ".jpeg"]])

# # 이미지 복사
# for p in tqdm(imgs, desc="Copy images"):
#     dst = PREP_ROOT / "images" / p.name
#     if not dst.exists():
#         dst.write_bytes(p.read_bytes())

# # 마스크 병합하고 저장
# for p in tqdm(imgs, desc="Build masks"):
#     stem = p.stem
#     img_id = int(stem)
#     merged = merge_masks_for_one(img_id, ORIG_ANNO_DIR)
#     Image.fromarray(merged).save(PREP_ROOT / "masks" / f"{stem}.png")

# print("Done:", PREP_ROOT)

Build masks: 100%|██████████| 30000/30000 [14:05<00:00, 35.50it/s]

Done: /home/jpsong/imgProcessing_Solo/CelebAMask-HQ


: 

: 

In [8]:
rng = np.random.default_rng(42)
stems = np.array([p.stem for p in (Path(DATA_ROOT)/"images").glob("*")])
rng.shuffle(stems)

val_ratio = 0.05
n_val = int(len(stems) * val_ratio)
val_ids = stems[:n_val]
train_ids = stems[n_val:]

(Path(DATA_ROOT)/"splits"/"train.txt").write_text("\n".join(train_ids.tolist())+"\n", encoding="utf-8")
(Path(DATA_ROOT)/"splits"/"val.txt").write_text("\n".join(val_ids.tolist())+"\n", encoding="utf-8")

print("train:", len(train_ids), "val:", len(val_ids))

train: 28500 val: 1500


In [ ]:
# =========================
# [ADD] Compute class weights from train masks
# =========================
def compute_class_pixel_counts(mask_dir: Path, ids, num_classes: int, ignore_index: int = 255):
    counts = np.zeros(num_classes, dtype=np.int64)
    for stem in tqdm(ids, desc="Count pixels(train masks)", leave=False):
        m = np.array(Image.open(mask_dir / f"{stem}.png"), dtype=np.int64)
        m = m[m != ignore_index]
        c = np.bincount(m.reshape(-1), minlength=num_classes)
        counts += c[:num_classes]
    return counts

train_mask_dir = Path(DATA_ROOT) / "masks"
train_pixel_counts = compute_class_pixel_counts(train_mask_dir, train_ids, NUM_CLASSES, IGNORE_INDEX)

freq = train_pixel_counts / (train_pixel_counts.sum() + 1e-12)

# median frequency balancing (background 제외해서 median 계산)
median_freq = np.median(freq[1:])  
weights = median_freq / (freq + 1e-12)

# weight 폭주 방지 (중요)
weights = np.clip(weights, 0.1, 10.0)

# background는 너무 영향 크니까 조금 낮춰도 됨(선택)
weights[0] = 0.3

class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

print("\n[Class weights]")
for i in range(NUM_CLASSES):
    print(f"{i:2d} {('background' if i==0 else LABELS[i-1]):12s} "
          f"count={train_pixel_counts[i]:>10d} weight={weights[i]:.3f}")
print()


Count pixels(train masks):  71%|███████   | 20302/28500 [00:36<00:14, 553.91it/s]

In [88]:
class CelebMaskHQPrepared(Dataset):
    def __init__(self, root: str, split: str, image_size: int, ignore_index: int = 255):
        self.root = Path(root)
        self.img_dir = self.root / "images"
        self.mask_dir = self.root / "masks"
        self.split_dir = self.root / "splits"
        self.split = split
        self.image_size = image_size
        self.ignore_index = ignore_index

        split_file = self.split_dir / f"{split}.txt"
        if split_file.exists():
            self.ids = [x.strip() for x in split_file.read_text(encoding="utf-8").splitlines() if x.strip()]
        else:
            self.ids = sorted([p.stem for p in self.img_dir.glob("*") if p.suffix.lower() in [".jpg",".png",".jpeg"]])

    def __len__(self):
        return len(self.ids)

    def _find_img_path(self, stem: str):
        for ext in [".jpg", ".png", ".jpeg"]:
            p = self.img_dir / f"{stem}{ext}"
            if p.exists():
                return p
        return None

    def __getitem__(self, idx):
        stem = self.ids[idx]
        img_path = self._find_img_path(stem)
        if img_path is None:
            raise FileNotFoundError(f"Image not found for {stem}")
        mask_path = self.mask_dir / f"{stem}.png"
        if not mask_path.exists():
            raise FileNotFoundError(f"Mask not found for {stem}")

        img = Image.open(img_path).convert("RGB")
        mask = np.array(Image.open(mask_path), dtype=np.int64)

        # Size 조정
        img = TF.resize(img, [self.image_size, self.image_size], interpolation=TF.InterpolationMode.BILINEAR)
        mask_pil = Image.fromarray(mask.astype(np.uint8))
        mask_pil = TF.resize(mask_pil, [self.image_size, self.image_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = np.array(mask_pil, dtype=np.int64)

        # train만 증가
        if self.split == "train" and torch.rand(1).item() < 0.5:
            img = TF.hflip(img)
            mask = np.ascontiguousarray(np.fliplr(mask))

        # to tensor + imagenet normalize
        img = TF.to_tensor(img)
        img = TF.normalize(img, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        mask = torch.from_numpy(mask).long()

        return img, mask, stem


train_ds = CelebMaskHQPrepared(DATA_ROOT, "train", IMAGE_SIZE, IGNORE_INDEX)
val_ds   = CelebMaskHQPrepared(DATA_ROOT, "val", IMAGE_SIZE, IGNORE_INDEX)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print("train:", len(train_ds), "val:", len(val_ds))


train: 28500 val: 1500


In [89]:
CLASS_NAMES = ["background"] + LABELS
assert NUM_CLASSES == len(CLASS_NAMES)

In [ ]:
from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights

def build_model(num_classes: int):
    weights = DeepLabV3_ResNet50_Weights.DEFAULT
    model = torchvision.models.segmentation.deeplabv3_resnet50(weights=weights)

    # 분류기 head 교체 (마지막 1x1 conv -> num_classes)
    if isinstance(model.classifier[-1], nn.Conv2d):
        in_ch = model.classifier[-1].in_channels
        model.classifier[-1] = nn.Conv2d(in_ch, num_classes, kernel_size=1)
    else:
        raise RuntimeError("분류기 버전 문제")

    # aux head도 있으면 교체
    if hasattr(model, "aux_classifier") and model.aux_classifier is not None:
        if isinstance(model.aux_classifier[-1], nn.Conv2d):
            in_ch = model.aux_classifier[-1].in_channels
            model.aux_classifier[-1] = nn.Conv2d(in_ch, num_classes, kernel_size=1)

    return model, weights

model, pretrained_weights = build_model(NUM_CLASSES)
model = model.to(device)

print("Loaded pretrained weights:", pretrained_weights)
print("Weights meta keys:", pretrained_weights.meta.keys())

Loaded pretrained weights: DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
Weights meta keys: dict_keys(['categories', 'min_size', '_docs', 'num_params', 'recipe', '_metrics', '_ops', '_file_size'])


In [91]:
@torch.no_grad()
def update_confmat(confmat: torch.Tensor, preds: torch.Tensor, gts: torch.Tensor,
                   num_classes: int, ignore_index: int):
    """
    confmat: [C,C] rows=GT, cols=Pred
    preds/gts: [N,H,W]
    """
    mask = (gts != ignore_index)
    gts = gts[mask].view(-1)
    preds = preds[mask].view(-1)

    k = (gts >= 0) & (gts < num_classes)
    gts = gts[k]
    preds = preds[k]

    inds = num_classes * gts + preds
    cm = torch.bincount(inds, minlength=num_classes**2).reshape(num_classes, num_classes)
    confmat += cm

def compute_metrics_from_confmat(confmat: torch.Tensor, include_bg: bool):
    """
    Returns dict:
      overall: pixel_acc, mPA, mIoU, mDice
      per-class: iou, dice, pa_cls, support
    """
    confmat = confmat.float()
    tp = torch.diag(confmat)
    fp = confmat.sum(dim=0) - tp
    fn = confmat.sum(dim=1) - tp

    # 클래스별 지표
    iou = tp / (tp + fp + fn + 1e-7)
    dice = (2 * tp) / (2 * tp + fp + fn + 1e-7)
    pa_cls = tp / (tp + fn + 1e-7)          
    support = confmat.sum(dim=1)        

    # 전반적인 지표
    pixel_acc = tp.sum() / (confmat.sum() + 1e-7)

    start = 0 if include_bg else 1
    miou = iou[start:].mean().item()
    mdice = dice[start:].mean().item()
    mpa = pa_cls[start:].mean().item()

    return {
        "pixel_acc": float(pixel_acc.item()),
        "mPA": float(mpa),
        "mIoU": float(miou),
        "mDice": float(mdice),
        "iou": iou.detach().cpu().numpy(),
        "dice": dice.detach().cpu().numpy(),
        "pa_cls": pa_cls.detach().cpu().numpy(),
        "support": support.detach().cpu().numpy(),
    }


In [92]:
def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.parameters():
        p.requires_grad = trainable

def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    total = 0.0

    for imgs, masks, _ in tqdm(loader, desc="train", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=(USE_AMP and device.type=="cuda")):
            logits = model(imgs)["out"]
            loss = criterion(logits, masks)

        if scaler is not None and scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total += loss.item() * imgs.size(0)

    return total / max(1, len(loader.dataset))

@torch.no_grad()
def evaluate(model, loader, include_bg_in_means=False):
    model.eval()
    confmat = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64, device=device)

    for imgs, masks, _ in tqdm(loader, desc="val", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits = model(imgs)["out"]
        preds = torch.argmax(logits, dim=1)
        update_confmat(confmat, preds, masks, NUM_CLASSES, IGNORE_INDEX)

    return compute_metrics_from_confmat(confmat, include_bg_in_means)

from pathlib import Path

def save_metrics_txt(metrics: dict, out_txt_path: str, class_names: list,
                     include_bg_in_means: bool, epoch: int | None = None):
    out_txt_path = Path(out_txt_path)
    out_txt_path.parent.mkdir(parents=True, exist_ok=True)

    with out_txt_path.open("a", encoding="utf-8") as f:
        f.write("=" * 90 + "\n")
        if epoch is not None:
            f.write(f"Epoch: {epoch}\n")
        f.write(f"Include background in mean metrics: {include_bg_in_means}\n")
        f.write(f"Pixel Accuracy: {metrics['pixel_acc']:.6f}\n")
        f.write(f"mPA:           {metrics['mPA']:.6f}\n")
        f.write(f"mIoU:          {metrics['mIoU']:.6f}\n")
        f.write(f"mDice:         {metrics['mDice']:.6f}\n\n")

        f.write(f"{'id':>3}  {'class':<12}  {'support(px)':>12}  {'IoU':>10}  {'Dice':>10}  {'PA':>10}\n")
        f.write("-" * 90 + "\n")

        for cid in range(NUM_CLASSES):
            name = class_names[cid] if cid < len(class_names) else f"class_{cid}"
            f.write(
                f"{cid:>3}  {name:<12}  {int(metrics['support'][cid]):>12}  "
                f"{metrics['iou'][cid]:>10.6f}  {metrics['dice'][cid]:>10.6f}  {metrics['pa_cls'][cid]:>10.6f}\n"
            )
        f.write("\n")


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE_INDEX)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler(enabled=(USE_AMP and device.type=="cuda"))

best_miou = -1.0

In [94]:
report_path = str(Path(OUT_DIR) / "eval_report.txt")

for epoch in range(1, EPOCHS+1):
    if epoch <= FREEZE_BACKBONE_EPOCHS:
        set_backbone_trainable(model, False)
    else:
        set_backbone_trainable(model, True)

    tr_loss = train_one_epoch(model, train_loader, optimizer, scaler, criterion)

    metrics = evaluate(model, val_loader, include_bg_in_means=False)
    save_metrics_txt(metrics, report_path, CLASS_NAMES, include_bg_in_means=False, epoch=epoch)

    msg = (f"[Epoch {epoch:03d}] loss={tr_loss:.4f} "
           f"mIoU={metrics['mIoU']:.4f} mDice={metrics['mDice']:.4f} "
           f"pixAcc={metrics['pixel_acc']:.4f} mPA={metrics['mPA']:.4f}")
    print(msg)

    # best 기준은 mIoU로 설정
    ckpt = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_miou": best_miou,
        "num_classes": NUM_CLASSES
    }
    torch.save(ckpt, str(Path(OUT_DIR)/"last.pt"))

    if metrics["mIoU"] > best_miou:
        best_miou = metrics["mIoU"]
        ckpt["best_miou"] = best_miou
        torch.save(ckpt, str(Path(OUT_DIR)/"best.pt"))
        print("  -> saved best.pt")


[Epoch 001] loss=0.2945 mIoU=0.4972 mDice=0.6079 pixAcc=0.9206 mPA=0.5991
  -> saved best.pt


[Epoch 002] loss=0.2141 mIoU=0.5043 mDice=0.6123 pixAcc=0.9226 mPA=0.6131
  -> saved best.pt


[Epoch 003] loss=0.1784 mIoU=0.5696 mDice=0.6768 pixAcc=0.9376 mPA=0.6764
  -> saved best.pt


[Epoch 004] loss=0.1580 mIoU=0.6041 mDice=0.7094 pixAcc=0.9405 mPA=0.6966
  -> saved best.pt


[Epoch 005] loss=0.1469 mIoU=0.6315 mDice=0.7369 pixAcc=0.9448 mPA=0.7269
  -> saved best.pt


...

[Epoch 036] loss=0.0611 mIoU=0.7263 mDice=0.8239 pixAcc=0.9526 mPA=0.8184


[Epoch 037] loss=0.0596 mIoU=0.7094 mDice=0.8110 pixAcc=0.9519 mPA=0.8035


[Epoch 038] loss=0.0585 mIoU=0.7160 mDice=0.8148 pixAcc=0.9523 mPA=0.8080


[Epoch 039] loss=0.0577 mIoU=0.7063 mDice=0.8091 pixAcc=0.9514 mPA=0.8035


[Epoch 040] loss=0.0562 mIoU=0.7228 mDice=0.8208 pixAcc=0.9527 mPA=0.8151


In [95]:
ckpt_path = Path(OUT_DIR)/"best.pt"
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()
print("Loaded:", ckpt_path, "epoch:", ckpt["epoch"], "best_miou:", ckpt["best_miou"])

/tmp/ipykernel_1038835/2026891907.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


Loaded: runs_celebamaskhq_hyperparam_tuning/best.pt epoch: 10 best_miou: 0.7324312329292297


In [96]:
final_report_path = str(Path(OUT_DIR) / "eval_report_final.txt")
final_metrics = evaluate(model, val_loader, include_bg_in_means=False)
save_metrics_txt(final_metrics, final_report_path, CLASS_NAMES, include_bg_in_means=False, epoch=ckpt["epoch"])
print("Final report saved:", final_report_path)

Final report saved: runs_celebamaskhq_hyperparam_tuning/eval_report_final.txt


In [97]:
import matplotlib.pyplot as plt

def simple_colormap(mask: np.ndarray, seed: int = 0):
    rng = np.random.default_rng(seed)
    max_id = int(mask.max())
    cmap = rng.integers(0, 255, size=(max_id+1, 3), dtype=np.uint8)
    cmap[0] = np.array([0,0,0], dtype=np.uint8)
    return cmap[mask.clip(0, max_id)]

@torch.no_grad()
def visualize_samples(n=3, save_dir=None):
    if save_dir is not None:
        Path(save_dir).mkdir(parents=True, exist_ok=True)

    model.eval()
    count = 0
    for imgs, masks, stems in val_loader:
        imgs = imgs.to(device)
        logits = model(imgs)["out"]
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        gts = masks.numpy()

        for i in range(preds.shape[0]):
            pred_color = simple_colormap(preds[i])
            gt_color = simple_colormap(gts[i])

            fig = plt.figure(figsize=(10,4))
            ax1 = fig.add_subplot(1,2,1)
            ax2 = fig.add_subplot(1,2,2)
            ax1.set_title("GT mask (colorized)")
            ax2.set_title("Pred mask (colorized)")
            ax1.imshow(gt_color)
            ax2.imshow(pred_color)
            ax1.axis("off"); ax2.axis("off")
            plt.tight_layout()

            if save_dir is not None:
                out_path = Path(save_dir)/f"{stems[i]}_gt_pred.png"
                fig.savefig(out_path, dpi=150)
                plt.close(fig)
            else:
                plt.show()

            count += 1
            if count >= n:
                return

visualize_samples(n=5, save_dir=str(Path(OUT_DIR)/"sample_vis"))
print("saved to:", Path(OUT_DIR)/"sample_vis")


saved to: runs_celebamaskhq_hyperparam_tuning/sample_vis
